<a href="https://colab.research.google.com/github/nomura-study-001/aistudy/blob/main/Colab_%E3%81%B8%E3%82%88%E3%81%86%E3%81%93%E3%81%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai -q

import os
from google.colab import userdata
from google import genai
from google.genai import types

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents='「AI職務経歴書エージェント」の開発に向けて、前向きなメッセージを200字程度でください。',
    config=types.GenerateContentConfig(
        system_instruction="あなたはIT業界に精通したキャリアアドバイザーです。"
    )
)
print(response.text)

「AI職務経歴書エージェント」の開発、心から応援しています！

転職活動において、自身の強みを言語化する「職務経歴書の作成」は、多くの求職者が最も苦戦する難所です。ここにAIの力を吹き込むことで、埋もれていた優秀な人材のポテンシャルが可視化され、IT業界の採用市場に大きな変革が起きると確信しています。

求職者と企業の未来を繋ぐ、革新的なプロダクトの誕生を心待ちにしています。開発頑張ってください！


In [3]:
# =====================================================================
# ① 準備：ライブラリのインポートと定数の定義
# =====================================================================
import os
import time  # 503エラー時のウェイト（待機時間）処理に使用
from google.colab import userdata
from google import genai
from google.genai import types
from google.genai.errors import APIError  # Gemini API特有のエラーを捕捉するためにインポート

# 試行回数などの設定を定数として定義
MAX_RETRIES = 3    # 503エラー時の最大リトライ回数
RETRY_DELAY = 5    # リトライまでの待機時間（秒）

# =====================================================================
# ② 安全なAPIキーの取得とクライアントの初期化（防御的プログラミング）
# =====================================================================
# 1日目に設定した「シークレット（左側の鍵マーク）」からAPIキーを取得
api_key = userdata.get('GEMINI_API_KEY')

# 【Claudeのアドバイス反映】APIキーが未設定、または大文字小文字のミスで取得できない場合は即座に処理を止める
if not api_key:
    raise ValueError("APIキーが取得できません。Colabのシークレットに「GEMINI_API_KEY」が正しく登録されているか確認してください。")

# Gemini APIと通信するための窓口（クライアント）を生成
client = genai.Client(api_key=api_key)

# =====================================================================
# ③ データの準備とプロンプトの作成（データの事前クレンジング）
# =====================================================================
# 分析対象となる仮の経歴データ（トリプルクォートによる前後の不要な改行を含む）
dummy_resume_raw = """
【氏名】開発 太郎
【年齢】50代
【職種】ITエンジニア（インフラ・社内SE中心）
【経験スキル】
- VBAを用いた業務効率化ツールの開発・保守（経験10年）
- 社内サーバー（Windows/Linux）の運用保守、ネットワーク構築
- ユーザーサポート、IT資産管理
【自己PR】
既存のVBA資産を活かした業務自動化が得意ですが、今後はPythonやAI技術を組み合わせた、より高度なエージェント開発に挑戦したいと考えています。
"""

# AIへの質問内容
user_question_raw = "私の経歴を踏まえて、今後『AIエージェント開発』に挑戦する際、どのような強みが活かせそうか客観的に分析してください。"

# 【Geminiのアドバイス反映】外部データや入力値の前後にある不要な改行・空白を削除（プロンプトの崩れを防止）
dummy_resume = dummy_resume_raw.strip()
user_question = user_question_raw.strip()

# 【ChatGPTのアドバイス反映】f文字列を用いて、文章構造を保ったまま経歴データと質問を変数として直感的に埋め込む
prompt = f"""
以下の[求職者の経歴データ]を読み込み、[質問]に対してキャリアアドバイザーの視点で回答してください。
必ず提供された経歴データの内容に基づいた具体的な回答にしてください。

[求職者の経歴データ]
{dummy_resume}

[質問]
{user_question}
"""

# =====================================================================
# ④ エラーハンドリングを備えた Gemini API の呼び出し（自動リトライ＆フォールバック）
# =====================================================================
# 【Claudeのアドバイス反映】503エラー（サーバー混雑）対策として、 try-except とリトライループを実装
response = None

for attempt in range(1, MAX_RETRIES + 1):
    try:
        # まずは最新の第一候補モデルでリクエストを送信
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction="あなたはIT業界に精通し、エンジニアのリスキリングを応援するキャリアアドバイザーです。",
                temperature=0.7  # 応答の多様性と創造性のバランスを調整（0.0で固定、1.0で最大ランダム）
            )
        )
        # 成功したらループを抜ける
        break

    except APIError as e:
        # エラーコードが503（Service Unavailable）の場合の処理
        if e.code == 503:
            print(f"[警告] Gemini APIが混雑しています（503エラー）。 {attempt}/{MAX_RETRIES} 回目の再試行を {RETRY_DELAY} 秒後に行います...")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                # 【Claudeのアドバイス反映】指定回数リトライしてもダメな場合、前世代の安定モデルへ切り替える（フォールバック）
                print("[情報] 規定回数リトライに失敗したため、代替モデル（gemini-1.5-flash）へ切り替えて再実行します。")
                try:
                    response = client.models.generate_content(
                        model='gemini-1.5-flash',
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            system_instruction="あなたはIT業界に精通し、エンジニアのリスキリングを応援するキャリアアドバイザーです。",
                            temperature=0.7
                        )
                    )
                except Exception as fatal_error:
                    print(f"[エラー] 代替モデルでの呼び出しにも失敗しました: {fatal_error}")
                    raise
        else:
            # 503以外のエラー（認証エラーなど）はリトライせずにその場で例外を発生させる
            print(f"[エラー] APIエラーが発生しました (コード: {e.code}): {e}")
            raise

# =====================================================================
# ⑤ 結果の出力
# =====================================================================
print("\n=== 分析結果 ===")
if response:
    print(response.text)
else:
    print("AIからの応答を取得できませんでした。")

[警告] Gemini APIが混雑しています（503エラー）。 1/3 回目の再試行を 5 秒後に行います...

=== 分析結果 ===
開発太郎さん、こんにちは！キャリアアドバイザーの視点から、太郎さんのこれまでの素晴らしいご経歴を拝見しました。

50代での「AIエージェント開発」という最先端分野への挑戦、非常に素晴らしいですね！IT業界が急速に変化する中、これまでの経験を土台にリスキリングを図ろうとする姿勢は、多くのエンジニアの模範となります。

太郎さんの経歴（VBA開発、インフラ、社内SE）は、一見するとモダンなAI開発とは距離があるように思えるかもしれませんが、実は**「実用的なAIエージェントをビジネス現場に実装する」という観点において、非常に強力なアドバンテージ（強み）**を持っています。

客観的に分析した、太郎さんならではの3つの強みを具体的にお伝えします。

---

### 強み1：10年のVBA経験で培った「業務プロセスの解剖力」と「自動化の設計力」
AIエージェントの究極の目的は「人間の代わりに判断し、業務を自動実行すること」です。
*   **活かせる強み：**
    太郎さんには、VBAを用いて10年間、現場の業務効率化に向き合ってきた実績があります。「どの業務が自動化に適しているか」「どのようなルールでデータが処理されているか」を瞬時に見抜く**業務プロセスの分析力（要件定義力）**は、一朝一夕では身につきません。
*   **AIエージェント開発への応用：**
    AIエージェントに「何を、どういう手順で考えさせ、実行させるか（エージェントのワークフロー設計）」を組み立てる際、この経験がそのまま活きます。また、既存のVBAマクロ（Excel等）とPythonで記述したAIエージェントを連携させ、**「レガシーシステムと最新AIを融合させた超効率化ツール」**を開発できるのは、太郎さんにしかできない独自の強みになります。

### 強み2：インフラ・サーバー運用の知見による「セキュアな実行環境の構築力」
AIエージェント（特にLLMなどの大規模言語モデルを活用するもの）は、APIの連携やサーバー上での実行環境の構築が不可欠です。
*   **活かせる強み：**
    Windows/Linuxサーバーの運用保守やネット